# Lab 7 - ABC del aprendizaje de máquina

Solución completa del laboratorio sobre el dataset de viviendas de California.

Este notebook responde los puntos solicitados: análisis exploratorio, limpieza, partición estratificada, codificación categórica, escalamiento, modelado y automatización con pipelines.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1.0 Análisis del data frame

Se carga el archivo desde la URL indicada en el laboratorio y se hace una inspección inicial del conjunto de datos.

In [2]:
url = "https://raw.githubusercontent.com/hernansalinas/Curso_aprendizaje_estadistico/main/datasets/Sesion_07_housing.csv"
df = pd.read_csv(url)

display(df.head())
print("Shape:", df.shape)
print("Columnas y tipos:")
print(df.dtypes)
print("Información general:")
df.info()
print("Estadísticos descriptivos:")
display(df.describe(include="all"))
print("Valores faltantes por columna:")
display(df.isnull().sum())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


Shape: (20640, 10)
Columnas y tipos:
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity           str
dtype: object
Información general:
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null 

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<1H OCEAN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9136
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909,NaN
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874,NaN
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000,NaN
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000,NaN
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000,NaN
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000,NaN


Valores faltantes por columna:


longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

### Respuestas a los puntos 3 y 4

- El dataset tiene 10 variables: 9 predictoras y 1 variable objetivo.
- Hay variables numéricas de tipo entero y flotante, y una variable categórica: `ocean_proximity`.
- Los valores faltantes se identifican con `isnull().sum()`; en este problema usualmente aparecen en `total_bedrooms`.
- `isnull()` e `isna()` son equivalentes en pandas.

In [ ]:
print("Elementos únicos en ocean_proximity:")
print(df["ocean_proximity"].unique())

cols = ["housing_median_age", "total_rooms", "total_bedrooms", "population", "households", "median_income", "median_house_value"]

print("Promedios por ocean_proximity:")
display(df.groupby("ocean_proximity")[cols].mean())

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.ravel()
for i, col in enumerate(cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Histograma de {col}")
for j in range(len(cols), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

df.boxplot(column="median_house_value", by="ocean_proximity", sym="k.", figsize=(18, 6))
plt.title("Boxplot for comparing price per living space for each city")
plt.suptitle("")
plt.xticks(rotation=20)
plt.show()

SyntaxError: unterminated string literal (detected at line 6) (3166632016.py, line 6)

### Respuesta a los puntos 5, 6, 7 y 8

- `ocean_proximity` contiene las categorías espaciales del distrito respecto al océano.
- El `groupby` permite comparar el comportamiento promedio de las variables numéricas dentro de cada categoría.
- Los histogramas muestran la distribución de cada variable y ayudan a detectar asimetrías y outliers.
- El boxplot permite comparar la distribución del precio medio por categoría espacial y detectar valores atípicos.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
display(corr_matrix)

plt.figure(figsize=(11, 7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Matriz de correlación")
plt.show()

pair_cols = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]
sns.pairplot(df[pair_cols], diag_kind="hist")
plt.show()

plt.figure(figsize=(10, 7))
sns.scatterplot(data=df, x="longitude", y="latitude", hue="median_house_value", palette="viridis", alpha=0.7, linewidth=0)
plt.title("Scatter plot coloreado por median_house_value")
plt.show()

### Respuesta al punto 9, 10 y 11

- La matriz de correlación resume relaciones lineales entre variables numéricas.
- `pairplot` permite observar relaciones bivariadas y la distribución marginal.
- El `scatter plot` espacial ayuda a visualizar cómo cambia el valor de la vivienda con la ubicación geográfica.

## 2.0 Preparación del data frame

Primero se analiza una partición aleatoria y luego se corrige el sesgo de muestreo con estratificación por rangos de ingreso.

In [ ]:
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)
print("Partición aleatoria")
print("train_set:", len(train_set))
print("test_set:", len(test_set))

df["income_cat"] = pd.cut(
    df["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(df, df["income_cat"]):
    housing_train = df.loc[train_index].copy()
    housing_test = df.loc[test_index].copy()

def income_cat_proportions(data):
    return data["income_cat"].value_counts(normalize=True).sort_index()

compare_props = pd.DataFrame({
    "Overall": income_cat_proportions(df),
    "Stratified Test": income_cat_proportions(housing_test),
    "Random Test": income_cat_proportions(test_set.assign(income_cat=pd.cut(test_set["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5])))
})
compare_props["Rand. %error"] = (100 * compare_props["Random Test"] / compare_props["Overall"] - 100).abs()
compare_props["Strat. %error"] = (100 * compare_props["Stratified Test"] / compare_props["Overall"] - 100).abs()
display(compare_props)

housing_train = housing_train.drop(columns=["income_cat"])
housing_test = housing_test.drop(columns=["income_cat"])

print("housing_train shape:", housing_train.shape)
print("housing_test shape:", housing_test.shape)

### Respuesta al punto 12 y 13

- La partición aleatoria es válida solo como primera aproximación, pero puede desbalancear variables importantes.
- La estratificación por categorías de ingreso conserva mejor la distribución original del dataset.
- `housing_train` y `housing_test` quedan como los dos data frames principales con características y etiqueta.

In [ ]:
# Feature engineering
for data in (housing_train, housing_test):
    data["rooms_per_household"] = data["total_rooms"] / data["households"]
    data["bedrooms_per_room"] = data["total_bedrooms"] / data["total_rooms"]
    data["population_per_household"] = data["population"] / data["households"]

# Separar características y etiqueta
train_features = housing_train.drop(columns=["median_house_value"])
train_labels = housing_train["median_house_value"].copy()
test_features = housing_test.drop(columns=["median_house_value"])
test_labels = housing_test["median_house_value"].copy()

num_cols = train_features.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = ["ocean_proximity"]

num_imputer = SimpleImputer(strategy="median")
train_num = pd.DataFrame(num_imputer.fit_transform(train_features[num_cols]), columns=num_cols, index=train_features.index)
test_num = pd.DataFrame(num_imputer.transform(test_features[num_cols]), columns=num_cols, index=test_features.index)

print("Medianas usadas por el imputador:")
display(pd.Series(num_imputer.statistics_, index=num_cols))
print("Medianas originales del training:")
display(train_features[num_cols].median())

### Respuesta al punto 14

El imputador usa la mediana de cada variable numérica para reemplazar los valores faltantes.
Comparar `imp_mean.statistics_` con `df_train_num.median()` permite verificar que el valor imputado coincide con el estadístico usado para limpiar los datos.

In [ ]:
cat_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
train_cat = pd.DataFrame(
    cat_encoder.fit_transform(train_features[cat_cols]),
    columns=cat_encoder.get_feature_names_out(cat_cols),
    index=train_features.index
)
test_cat = pd.DataFrame(
    cat_encoder.transform(test_features[cat_cols]),
    columns=cat_encoder.get_feature_names_out(cat_cols),
    index=test_features.index
)

train_prepared = pd.concat([train_num, train_cat], axis=1)
test_prepared = pd.concat([test_num, test_cat], axis=1)

scaler = MinMaxScaler()
train_prepared_scaled = pd.DataFrame(scaler.fit_transform(train_prepared), columns=train_prepared.columns, index=train_prepared.index)
test_prepared_scaled = pd.DataFrame(scaler.transform(test_prepared), columns=test_prepared.columns, index=test_prepared.index)

display(train_cat.head())
display(train_prepared.head())
display(train_prepared_scaled.head())

### Respuesta a los puntos 15 y 16

- `OneHotEncoder` convierte la variable categórica `ocean_proximity` en columnas binarias.
- `MinMaxScaler` reescala las variables numéricas a un rango comparable, generalmente entre 0 y 1.
- El resultado final es un conjunto de características listo para modelado.

In [ ]:
# Modelo base: regresión lineal
lin_reg = LinearRegression()
lin_reg.fit(train_prepared_scaled, train_labels)

train_pred = lin_reg.predict(train_prepared_scaled)
test_pred = lin_reg.predict(test_prepared_scaled)

train_r2 = r2_score(train_labels, train_pred)
test_r2 = r2_score(test_labels, test_pred)
train_mae = mean_absolute_error(train_labels, train_pred)
test_mae = mean_absolute_error(test_labels, test_pred)
train_rmse = mean_squared_error(train_labels, train_pred, squared=False)
test_rmse = mean_squared_error(test_labels, test_pred, squared=False)

print("Regresión lineal")
print("R2 train:", train_r2)
print("R2 test :", test_r2)
print("MAE train:", train_mae)
print("MAE test :", test_mae)
print("RMSE train:", train_rmse)
print("RMSE test :", test_rmse)

# Modelo alternativo
rf_reg = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_reg.fit(train_prepared_scaled, train_labels)
rf_train_r2 = rf_reg.score(train_prepared_scaled, train_labels)
rf_test_r2 = rf_reg.score(test_prepared_scaled, test_labels)
print("Random Forest")
print("R2 train:", rf_train_r2)
print("R2 test :", rf_test_r2)

### Respuesta a los puntos 1, 2, 3 y 4

1. El modelo lineal sirve como línea base, pero normalmente no captura toda la complejidad del problema de vivienda.
2. La regresión lineal es válida como primer modelo, aunque el comportamiento espacial y las relaciones no lineales suelen limitar su capacidad.
3. El `score` en regresión es el coeficiente de determinación $R^2$; mide cuánta variabilidad de la variable objetivo explica el modelo.
4. Sí, puede ajustarse a otros modelos como `RandomForestRegressor`, `GradientBoostingRegressor` o modelos regularizados como `Ridge`.

Una buena señal es que el puntaje de prueba sea cercano al de entrenamiento; si el puntaje de entrenamiento es mucho mayor, hay sobreajuste.

In [ ]:
# Automatización completa con pipelines
numeric_features = train_features.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = ["ocean_proximity"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

X_train = housing_train.drop(columns=["median_house_value"])
y_train = housing_train["median_house_value"]
X_test = housing_test.drop(columns=["median_house_value"])
y_test = housing_test["median_house_value"]

full_pipeline.fit(X_train, y_train)
pipeline_train_r2 = full_pipeline.score(X_train, y_train)
pipeline_test_r2 = full_pipeline.score(X_test, y_test)

print("Pipeline con regresión lineal")
print("R2 train:", pipeline_train_r2)
print("R2 test :", pipeline_test_r2)

pred_pipeline = full_pipeline.predict(X_test)
print("MAE test :", mean_absolute_error(y_test, pred_pipeline))
print("RMSE test :", mean_squared_error(y_test, pred_pipeline, squared=False))

### Respuesta al punto 5

El proceso se automatiza usando `Pipeline` y `ColumnTransformer`.

Esto permite encadenar imputación, escalamiento, codificación categórica y modelo en un solo objeto reproducible.

Ventajas principales:
- evita fugas de información;
- simplifica el entrenamiento y la evaluación;
- facilita el cambio de modelo sin rehacer el preprocesamiento;
- mejora la reproducibilidad del experimento.

Conclusión general: la regresión lineal es una buena referencia inicial, pero normalmente se pueden obtener mejores resultados con modelos no lineales y con una ingeniería de variables más rica.